# Análise exploratória de dados
 - Conjunto de dados: `churn` (Taxa de cancelamento/rotatividade de clientes de um banco)
 - Cientistas de dados:
   - Fábio Agostinho(fabinhosnf@gmail.com)
   - Isabella Almeida (isabella4lmeidafernandes@gmail.com)
   - Marcos Felipe Santos (marcosfelipessc@alu.ufc.br)
---

## Preparação
 - Carregamento de bibliotecas
 - Configuração de estilo dos gráficos
 - Leitura do conjunto de dados

In [ ]:
import itertools
import pandas as pd
import numpy as np
from scipy import stats as st
from matplotlib import pyplot as plt
import seaborn as sns
from IPython.display import Markdown
import math

# Tema global dos gráficos: Fundo branco, grade ativada, paleta de cores amigável a pessoas com daltonismo
sns.set_theme(style="whitegrid", palette="colorblind")

In [ ]:
# @title Leitura do conjunto de dados
df = pd.read_csv(
    "https://raw.githubusercontent.com/omadson/datasets/refs/heads/main/datasets/churn.csv"
)

HTTPError: HTTP Error 404: Not Found

In [ ]:
# @title Informações iniciais
display(Markdown("### Primeiras linhas"))
display(df.head())

display(Markdown("### Ultimas linhas"))
display(df.tail())

display(Markdown("### Informação das variáveis"))
df.info()

display(Markdown("### Quantidade de valores únicos"))
df.nunique()

---
A partir das informações iniciais, podemos dizer que:
 - O conjunto de dados tem 10.000 unidades amostrais com 14 variáveis
 - Classificação das variáveis:
    - Quantitativa contínua: `CreditScore`, `Balance`, `EstimatedSalary`
    - Quantitativa discreta: `RowNumber`, `Age`, `Tenure`, `NumOfProducts`
    - Qualitativa nominal: `CustomerId`, `Surname`, `Geography`, `Gender`, `HasCrCard`, `IsActiveMember`, `Exited`
---

In [ ]:
from typing import AsyncGenerator
# @title Dicionário de dados
df_dict = pd.DataFrame([
    {
        "variavel": "RowNumber",
        "descricao": "O número sequencial atribuído a cada linha no conjunto de dados.",
        "tipo": "quantitativa",
        "subtipo": "discreta",
    },
    {
        "variavel": "CustomerId",
        "descricao": "Um identificador único para cada cliente.",
        "tipo": "qualitativa",
        "subtipo": "nominal",
    },
    {
        "variavel": "Surname",
        "descricao": "O sobrenome do cliente.",
        "tipo": "qualitativa",
        "subtipo": "nominal",
    },
    {
        "variavel": "CreditScore",
        "descricao": "A pontuação de crédito do cliente.",
        "tipo": "quantitativa",
        "subtipo": "contínua",
    },
    {
        "variavel": "Geography",
        "descricao": "A localização geográfica do cliente (país).",
        "tipo": "qualitativa",
        "subtipo": "nominal",
    },
    {
        "variavel": "Gender",
        "descricao": "O gênero do cliente (feminino, masculino etc.)",
        "tipo": "qualitativa",
        "subtipo": "nominal",
    },
    {
        "variavel": "Age",
        "descricao": "A idade do cliente.",
        "tipo": "quantitativa",
        "subtipo": "discreta",
    },
    {
        "variavel": "Tenure",
        "descricao": "O número de anos que o cliente é correntista do banco.",
        "tipo": "quantitativa",
        "subtipo": "discreta",
    },
    {
        "variavel": "Balance",
        "descricao": "O saldo da conta do cliente.",
        "tipo": "quantitativa",
        "subtipo": "contínua",
    },
    {
        "variavel": "NumOfProducts",
        "descricao": "O número de produtos bancários que o cliente possui.",
        "tipo": "quantitativa",
        "subtipo": "discreta",
    },
    {
        "variavel": "HasCrCard",
        "descricao": "Indica se o cliente possui um cartão de crédito (sim = 1/não = 0).",
        "tipo": "qualitativa",
        "subtipo": "nominal",
    },
    {
        "variavel": "IsActiveMember",
        "descricao": "Indica se o cliente é um membro ativo (sim = 1/não = 0).",
        "tipo": "qualitativa",
        "subtipo": "nominal",
    },
    {
        "variavel": "EstimatedSalary",
        "descricao": "O salário estimado do cliente.",
        "tipo": "quantitativa",
        "subtipo": "contínua",
    },
    {
        "variavel": "Exited",
        "descricao": "Indica se o cliente saiu do banco (sim = 1/não = 0).",
        "tipo": "qualitativa",
        "subtipo": "nominal",
    }
    ])
df_dict

#Análise Univariada

In [ ]:
# @title Resumo estatístico

display(Markdown("### Variáveis qualitativas"))
cols_qualitativas = ['Surname', 'Geography', 'Gender', 'HasCrCard', 'IsActiveMember', 'Exited']
print(df[cols_qualitativas].astype('object').describe())

display(Markdown("### Variáveis quantitativas"))
cols_quantitativas = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']
print(df[cols_quantitativas].describe())

---

- A maioria dos clientes são `homens`
- A média de idade dos clientes é  de `38,9 anos`
- A maior parte dos clientes são da `França`
- A grande maioria dos clientes `possui cartão de crédito`
- Pelo menos `metade dos clientes tem a conta ativa`, ou seja, realiza movimentações
- Apenas `20% dos clientes cancelou` o serviço com o banco

---




#Distribuição de Variáveis

In [ ]:
# @title Variáveis qualitativas
variaveis_qualitativas = ['Geography', 'Gender', 'HasCrCard', 'IsActiveMember', 'Exited']

fig, axes = plt.subplots(figsize=(15, 8), ncols=3, nrows=2)
axes = axes.flatten()

# Dicionário de mapeamento para variáveis binárias (0 e 1)
mapa_legenda = {0: 'Não', 1: 'Sim'}

# Criar cópia temporária da coluna para o gráfico

for i, variavel in enumerate(variaveis_qualitativas):
    if variavel in ['HasCrCard', 'IsActiveMember', 'Exited']:
        dados_para_plotar = df[variavel].map(mapa_legenda)
    else:
        dados_para_plotar = df[variavel]

    # Criar a figura
    ax = sns.countplot(df, y=dados_para_plotar, ax=axes[i], alpha=.8)
    ax.bar_label(ax.containers[0], fmt="%d", color="white", label_type="center")
    ax.set(title=f"Distribuição da variável {variavel}", xlabel="Quantidade")
    for side in ["bottom", "top", "right"]:
        ax.spines[side].set_visible(False)
    ax.spines["left"].set_color("black")

fig.delaxes(axes[5])
plt.tight_layout()
plt.show()


---

- A maioria dos clientes é da França (5.014), e o restante se divide em Alemanha (2.509) e Espanha (2.477)
- Os homens representam a maior parte dos clientes (5.457), enquanto as mulheres representam a menor parte (4.543)
- A grande maioria dos clientes possui cartão de crédito (7.055), e somente uma menor parte (2.945) não possui
- Mais da metade dos clientes é um membro ativo (5.151 vs. 4.849), ou seja, realiza movimentações em sua conta
- Apenas uma pequena parte dos clientes (2.037 vs. 7.963) cancelou a conta/os serviços com o banco

---

Variáveis Quantitativas

In [ ]:
variaveis_quantitativas = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']

# Criar a figura e o grid
fig = plt.figure(figsize=(16, 10))

outer_grid = fig.add_gridspec(2, 3, wspace=0.3, hspace=0.4)

for i, variable in enumerate(variaveis_quantitativas):
  inner_grid = outer_grid[i].subgridspec(2, 1, height_ratios=[3, 1], hspace=0.0)
  ax_hist = fig.add_subplot(inner_grid[0])
  ax_box = fig.add_subplot(inner_grid[1])

  # Plotagem do histograma
  sns.histplot(data=df, x=variable, ax=ax_hist, kde=True)
  ax_hist.set_title(variable, fontsize=12, fontweight='bold')
  ax_hist.set_xlabel("")
  ax_hist.set_xticklabels([])
  ax_hist.set_ylabel("Quantidade")
  ax_hist.axvline(df[variable].mean(), color='k', label='Média')
  ax_hist.axvline(df[variable].median(), color='k', linestyle='--', label='Mediana')
  ax_hist.spines['top'].set_visible(False)
  ax_hist.spines['right'].set_visible(False)
  ax_hist.spines['bottom'].set_visible(False)

  if i == 0:
        ax_hist.legend(loc='upper right', fontsize='small')
  else:
      if ax_hist.legend_: ax_hist.legend_.remove()

  # Plotagem do boxplot
  sns.boxplot(data=df, x=variable, ax=ax_box)
  ax_box.set_xlabel(variable, fontsize=10)
  ax_box.set_yticks([])
  ax_box.spines['top'].set_visible(False)
  ax_box.spines['right'].set_visible(False)
  ax_box.spines['left'].set_visible(False)

plt.show()

---

Distribuição da variável `CreditScore`:
- Lembra muito uma curva de sino (normal)
- A maioria dos clientes tem score médio (entre 600 e 700)
- Há scores muito baixos (abaixo de 400), considerados anomalias

Distribuição da variável `Age`:
- É assimétrica para a direita
- Há uma maior concentração de clientes entre 30 e 40 anos
- Existem muitos clientes acima de 60 anos, sendo considerados anomalias

Distribuição da variável `Tenure`:
- É praticamente uniforme
- Há quantidades muito parecidas de clientes que estão no banco há 1 ano, 5 anos ou 9 anos

Distribuição da variável `Balance`:
- É bimodal
- No pico 1, tem-se uma grande parcela dos clientes com saldo zero (talvez mais de 30%)
- No pico 2, há uma distribuição normal para quem tem dinheiro na conta, centrada em torno de 100k-125k

Distribuição da variável `NumOfProducts`:
- A grande maioria dos clientes tem 1 ou 2 produtos
- Clientes com 3 ou 4 produtos são raríssimos, sendo situações consideradas anomalias

Distribuição da variável `EstimatedSalary`:
- Distribuição uniforme
- Todas as faixas de renda quase na mesma proporção

---

### Dados faltantes

In [ ]:
df.isnull().sum()

---

Não existem dados faltantes

---

#Análise Bivariada

Relação Entre Variáveis Quantitativas

Comportamento par a par

In [ ]:
# @title Relação Par a Par

vars_quant_bivariada = ['CreditScore', 'Age', 'Balance', 'EstimatedSalary', 'Exited']

g = sns.pairplot(
    df[vars_quant_bivariada],
    hue='Exited',
    diag_kind='kde',
    corner=True,
    palette=['#4e79a7', '#f28e2b'],
    height=2.5,
    plot_kws={
        's': 15,
        'alpha': 0.6,
        'edgecolor': 'none'
    },
    diag_kws={
        'fill': True,
        'alpha': 0.6
    }
)
g.fig.suptitle("Relação Par a Par das Variáveis Quantitativas", y=1.02, fontsize=16)


for ax in g.axes.flatten():
    if ax is not None:
        ax.tick_params(axis='x', rotation=30)

plt.show()

*   O fator idade é o discriminante mais claro. Observando a diagonal da variável Age, a curva laranja que representa os clientes que saíram está deslocada para a direita em relação à azul. Isso indica que clientes mais velhos (entre 40 e 50 anos) têm maior tendência ao churn do que os mais jovens.

*   No saldo bancário a distribuição é bimodal: nota-se que uma grande parte dos clientes que permanecem possui saldo zero. Por outro lado, o churn parece ser proporcionalmente maior entre clientes que possuem saldo positivo na conta.

*   As variáveis CreditScore e EstimatedSalary apresentam distribuições quase idênticas para ambos os grupos, considerando que as curvas azul e laranja se sobrepõem perfeitamente. Isso sugere que o salário e a pontuação de crédito, isoladamente, não são fatores determinantes para a saída do cliente.

Correlação

In [ ]:
# @title Correlação das variáveis quantitativas

colunas_interesse = ['CreditScore', 'Age', 'Balance', 'EstimatedSalary', 'Exited']
corr_matrox = df[colunas_interesse].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrox,
    annot = True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1, vmax=1
)
plt.title("Matrix de correlação")
plt.show()

* A Idade apresenta a maior correlação positiva com a saída, com um valor de 0.29. Estatisticamente, isso confirma que o envelhecimento do cliente é o principal fator numérico associado ao aumento do risco de churn.

* O Saldo em conta apresenta uma correlação positiva fraca, de apenas 0.12. Isso indica uma leve tendência de que clientes com maior acúmulo de capital no banco sejam mais propensos a sair do que aqueles com saldo baixo.

* O Salário Estimado e o Score de Crédito possuem correlações próximas de zero. Isso sugere que a decisão de sair do banco não depende linearmente de quanto o cliente ganha ou de quão bom pagador ele é.

Relação entre variáveis Qualitativas

Contingência

In [ ]:
# @title Contingência das variáveis qualitativas

vars_quali = ['Geography', 'Gender', 'IsActiveMember', 'HasCrCard']


fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, var in enumerate(vars_quali):


    ct = pd.crosstab(df[var], df['Exited'], normalize='index')


    ct.plot(kind='bar', stacked=True, ax=axes[i], color=['#4e79a7', '#f28e2b'], alpha=0.8)


    axes[i].set_title(f"Impacto de {var} no Churn", fontsize=12)
    axes[i].set_ylabel("Proporção")
    axes[i].set_xlabel("")
    axes[i].legend(title='Exited', loc='upper right')

    axes[i].axhline(y=0.2, color='red', linestyle='--', alpha=0.3, label='Média Global')

plt.suptitle("Análise Bivariada - Variáveis Qualitativas", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

* O país de residência é um discriminante fortíssimo. Enquanto França e Espanha mantêm taxas de saída abaixo da média global, a Alemanha apresenta uma taxa alarmante, superior a 30%. Isso indica um problema localizado no mercado alemão.

* Existe uma disparidade clara entre os sexos. Clientes do gênero Feminino têm uma probabilidade de saída consideravelmente maior do que os do gênero Masculino, superando a média global.

* Conforme esperado, o engajamento retém clientes. Membros inativos têm uma taxa de cancelamento muito superior à dos membros ativos.

* A variável de cartão de crédito mostrou-se neutra. As barras são praticamente idênticas para quem tem e quem não tem cartão, indicando que a posse desse produto não influencia na decisão de sair ou ficar."

Relação entre variáveis quantitativas e qualitativas

Distribuição relativa

In [ ]:
# @title Distribuição Relativa

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

sns.boxplot(
    x='Exited', y='Age', hue='Exited',
    data=df, ax=axes[0],
    palette=['#4e79a7', '#f28e2b'], legend=False
)
axes[0].set_title("Distribuição de Idade por Status")

sns.boxplot(
    x='Exited', y='Balance', hue='Exited',
    data=df, ax=axes[1],
    palette=['#4e79a7', '#f28e2b'], legend=False
)
axes[1].set_title("Distribuição de Saldo por Status")

sns.violinplot(
    x='Exited', y='NumOfProducts', hue='Exited',
    data=df, ax=axes[2],
    palette=['#4e79a7', '#f28e2b'], legend=False
)
axes[2].set_title("Número de Produtos por Status")

plt.tight_layout()
plt.show()


-   O gráfico afere visualmente que a idade é o fator mais crítico. A mediana de idade de quem sai gira em torno de 45 anos, enquanto a de quem fica é próxima de 36 anos. A caixa laranja está quase inteiramente posicionada acima da mediana da caixa azul, indicando uma separação clara entre os grupos.
-   O churn tende a ocorrer em clientes com saldo mais elevado. É possível notar que a caixa azul tem uma dispersão maior que inclui muitos saldos baixos ou zerados, enquanto a caixa laranja é mais compacta e concentrada em faixas de saldo mais altas. Isso é preocupante, pois o banco está perdendo clientes com capital.
-  O gráfico de número de produtos revela um comportamento complexo:
   - Zona de Risco 1: Clientes com apenas 1 produto representam uma grande parte do churn.
   - Zona de Segurança: Clientes com 2 produtos são extremamente fiéis.
   - Zona de Risco 2: Ter 3 ou 4 produtos é um indicador quase certo de saída.

#Análise Multivariada

In [ ]:
# @title Distribuição conjunta

variaveis_quantitativas_multi = [
    "CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "EstimatedSalary"
]

variaveis_qualitativas_multi = [
    "Geography", "Gender", "HasCrCard", "IsActiveMember"
]

g = sns.PairGrid(
    df,
    y_vars=variaveis_quantitativas_multi,
    x_vars=variaveis_qualitativas_multi,
    hue="Exited",
    height=2.5
)

g.fig.set_size_inches(12, 10)
g.map(sns.violinplot, split=True, inner="quart", gap=.1, cut=0)

plt.suptitle("Relação entre variáveis quantitativas e qualitativas")
plt.tight_layout()
g.add_legend()
plt.show()

---


*   Clientes que cancelaram (Exited=1) tendem a ser ligeiramente mais velhos que os que permaneceram, padrão consistente em diferentes países e gêneros.

*   Clientes com maior saldo (Balance) apresentam maior concentração entre os que cancelaram, independentemente de Geography ou Gender.

*   O número de produtos (NumOfProducts) mostra que clientes com apenas 1 produto concentram maior proporção de churn em comparação com aqueles com mais produtos.

*   Membros inativos (IsActiveMember=0) apresentam distribuições mais associadas ao churn do que membros ativos.

*   CreditScore e EstimatedSalary possuem distribuições muito semelhantes entre clientes que saíram e permaneceram, indicando baixo poder discriminativo conjunto.

*   Diferenças entre países (Geography) sugerem leve maior concentração de churn na Alemanha em diferentes níveis de idade e saldo.


---

In [ ]:
# @title Contingência das variáveis qualitativas

sns.set_style('whitegrid')

combinacoes = sorted(
    itertools.combinations(variaveis_qualitativas_multi, 2),
    key=lambda x: x[1]
)

fig, axes = plt.subplots(figsize=(12,7), ncols=3, nrows=2, squeeze=False)
axes = axes.flatten()

for i, (var_1, var_2) in enumerate(combinacoes):
    ax = sns.boxplot(
        df, x=var_1, y='Age', hue=var_2, gap=.1,
        ax=axes[i]
    )
    ax.set(title=f"{var_1} x {var_2} x Age")

    for side in ["left", "top", "right"]:
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color("black")

    sns.move_legend(
        axes[i], "lower center",
        bbox_to_anchor=(.5, -.6), ncol=3, title=None,
    )

plt.suptitle("Contingência das variáveis qualitativas")
plt.tight_layout()
plt.show()


---

*   A distribuição de idade é bastante semelhante entre homens e mulheres em todos os países, com leve aumento da mediana na Alemanha.

*   Clientes com cartão de crédito (HasCrCard=1) apresentam idades levemente maiores em média, independentemente do país.

*   A variável Gender não mostra diferenças relevantes na idade quando combinada com posse de cartão de crédito.

*   Membros ativos (IsActiveMember=1) tendem a ter idades ligeiramente maiores do que membros inativos em diferentes geografias.

*   A combinação de HasCrCard e IsActiveMember indica que clientes ativos com cartão concentram-se em faixas etárias um pouco mais elevadas.

*   No geral, as distribuições são muito próximas, sugerindo que idade não varia drasticamente entre as categorias qualitativas analisadas.

---

In [ ]:
# @title Matriz de correlação (apenas quantitativas)

plt.figure(figsize=(10,7))
corr = df[variaveis_quantitativas_multi].corr()

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlação entre Variáveis Quantitativas")
plt.show()


---


*   As variáveis quantitativas apresentam, em geral, correlações muito fracas entre si, indicando baixa multicolinearidade.

*   A maior relação observada é negativa entre Balance e NumOfProducts (-0,30), sugerindo que clientes com mais produtos tendem a manter saldos menores.

*   CreditScore, Age, Tenure e EstimatedSalary possuem correlações próximas de zero com as demais variáveis, indicando comportamentos relativamente independentes.

*   EstimatedSalary praticamente não se correlaciona com nenhuma outra variável, mostrando baixa associação linear com o perfil financeiro e de relacionamento do cliente.


---

In [ ]:
# @title PCA (somente variáveis quantitativas)

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

num_cols = [
    "CreditScore", "Age", "Tenure",
    "Balance", "NumOfProducts", "EstimatedSalary"
]

scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[num_cols])

pca = PCA(n_components=2)
pca_result = pca.fit_transform(scaled_data)

pca_df = pd.DataFrame(pca_result, columns=["PC1", "PC2"])
pca_df["Exited"] = df["Exited"].values  # usado só para visualização

plt.figure(figsize=(8,6))
sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Exited",
    alpha=0.6
)
plt.title("PCA: Projeção Multivariada dos Clientes")
plt.show()


In [ ]:
# DataFrame com os pesos (loadings) das variáveis em cada componente
loadings = pd.DataFrame(
    pca.components_,
    columns=variaveis_quantitativas,
    index=['PC1', 'PC2']
)

loadings

---
PC1
*   A projeção em PCA mostra grande sobreposição entre clientes que saíram (Exited=1) e os que permaneceram (Exited=0), indicando baixa separação linear entre os grupos.

*   Observa-se leve concentração de churn em regiões específicas do espaço projetado (principalmente valores mais altos de PC1), sugerindo influência conjunta de algumas variáveis.

*   A dispersão elevada dos pontos indica alta variabilidade no perfil dos clientes, sem clusters claramente definidos apenas com as componentes principais.

*   O PCA sugere que o churn não é explicado por uma única combinação linear dominante, sendo necessário considerar múltiplas variáveis simultaneamente para melhor discriminação.

PC2
*   A componente é dominada por EstimatedSalary (+0.64), Tenure (+0.58) e Age (-0.48).

*   Assim, o PC2 representa um eixo relacionado ao perfil de renda e tempo de relacionamento com o banco, em oposição à idade.

*   Valores altos em PC2 indicam clientes com maior salário estimado e maior tempo de permanência no banco, enquanto valores baixos estão associados a clientes mais velhos.

Insight Geral
*   O churn tende a se distribuir ao longo de dois perfis principais: um ligado à relação entre saldo e quantidade de produtos (PC1) e outro ligado a renda, tempo de relacionamento e idade (PC2).

*   Isso sugere que a evasão de clientes não depende de uma única variável isolada, mas de combinações entre perfil financeiro (saldo vs produtos) e perfil de relacionamento/renda (tenure, salário e idade).

---

#Sumário de Insights e Hipóteses

### **Quais padrões e tendências foram identificados?**
---
A análise de 10.000 observações identificou os seguintes determinantes de *churn*, fundamentando a etapa de modelagem preditiva:


* **Taxa de Churn:** Desbalanceamento moderado, com **20,4%** de evasão.
* **Demografia:** Idade média de 38,9 anos, maioria masculina (54%) e concentração geográfica na França (50%).

---

*  A idade tem forte relação linear positiva (correlação: 0.29), com escalada substancial de risco entre **40 e 60 anos**.
* A **Alemanha** apresenta anomalia regional severa, com taxa de *churn* superior a 30%.
* A retenção depende estritamente do volume de produtos:
    * **1 Produto:** *churn* elevado.
    * **2 Produtos:** Ponto ótimo de retenção.
    * **3+ Produtos:** Evasão próxima a 100%, indicando insatisfação ou oferta inadequada.
* O *churn* concentra-se em clientes com **saldo positivo e elevado**; contas zeradas apresentam maior retenção.
* **Fatores Irrelevantes:** `EstimatedSalary` e `CreditScore` não possuem correlação com a decisão de saída.

---
* Baixa colinearidade global, com exceção da relação inversa moderada (-0.30) entre `Balance` e `NumOfProducts`. A projeção PCA mostra alta sobreposição entre classes, sem hiperplano de separação linear.
* Devido à não-linearidade crítica (`NumOfProducts`) e anomalias locais, como a Alemanha, algoritmos lineares terão desempenho limitado. Recomenda-se o uso de modelos baseados em árvores (**Random Forest**, **Gradient Boosting/XGBoost**), capazes de capturar nativamente essas fronteiras de decisão complexas.

---

### **Há valores extremos ou dados faltantes que podem impactar a análise?**  

-  Não foram identificados dados faltantes. O conjunto de dados apresenta completude total em todas as 14 variáveis para as 10.000 observações, não sendo necesspario técnicas de imputação ou remoção de linhas.

Identificamos *outliers* estatísticos, mas que representam comportamentos reais e críticos para o negócio, sendo necessário permancerem na análise:

- `Age`:
    - O boxplot revela diversos pontos acima do limite superior, com clientes de idade superior à 62 anos.
    - O Impacto é alto e Informativo. Estes *pontos fora da curva* são sum nicho de clientes mais velhos com comportamento de risco distinto. Remover estes dados mascararia uma tendência importante de *churn*.

- `NumOfProducts`:
    - A categoria "4 produtos" é extremamente rara, considerada anomalia estatística pelo baixo volume.
    - O impacto é crítico. Apesar de serem *outliers* em frequência, representam um grupo com taxa de cancelamento de quase 100%. Devem ser tratados como um sinal de alerta para insatisfação ou erro de venda.

- `CreditScore`:
    -  Valores abaixo de 400 pontos aparecem como *outliers* inferiores.
    -  O impacto é baixo. Como a análise de correlação mostrou que o score não influencia a decisão de saída, esses valores extremos são irrelevantes para a modelagem do *churn*.

---

### **Existem correlações ou associações relevantes entre as variáveis?**

A análise revelou que existem correlações fortes, porém de naturezas distintas:

* **`Age` x Churn:** É a correlação positiva mais forte com a variável alvo. Estatisticamente, confirma que o envelhecimento do cliente aumenta a probabilidade linear de cancelamento.
* **`Balance` x `NumOfProducts`:** Identificou-se uma correlação negativa moderada. Clientes com muitos produtos tendem a ter menos saldo em conta. fruto de possível diluição de capital em investimentos ou seguros, enquanto clientes com apenas 1 produto concentram maior liquidez.
* **Baixa Correlação:** `EstimatedSalary` e `CreditScore` apresentaram correlação quase nula com o *churn*, indicando independência linear.

Embora não capturadas pela correlação de Pearson, existem associações comportamentais críticas:
* **`NumOfProducts`:** A relação é não-linear. A posse de **2 produtos** está fortemente associada à retenção, enquanto **3 ou 4 produtos** têm associação quase determinística com o cancelamento.
* **`Geography`:** Existe uma dependência espacial clara. A categoria *Alemanha* está associada a uma taxa de *churn* significativamente superior em comparação a França e Espanha.
* **`IsActiveMember`:** A inatividade é um previsor robusto de saída, confirmando a associação entre falta de interação bancária recente e cancelamento.